# IMPORT

In [1]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

import os
import gc
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [2]:
STATION_CLUSTER_FOLDER = Path("../data/station_clusters")
DATASET_FOLDER = Path("../data/clean_data_csv")
MODEL_OUTPUT_FOLDER = Path("../data/cluster_models")
PLOT_FOLDER = Path("../data/model_plots")

folders = [STATION_CLUSTER_FOLDER, DATASET_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [3]:
FEATURE_NAME = "11_features" # "11_features" or "5_features"
HAS_BIKE_LAG = True
LAGGED_NAME = "lagged" if HAS_BIKE_LAG else "not_lagged"
CLUSTER_FILE_NAME = f"station_clusters_2_42_{FEATURE_NAME}"

# LOAD DATA AND CLUSTERS

In [4]:
df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/{CLUSTER_FILE_NAME}.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

Rows: (6710838, 21)
Clusters: cluster
0    3433586
1    3277252
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'total_bikes_lag1',
       'total_bikes_lag2', 'split', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [5]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",

    # Station context
    "capacity"
]
if HAS_BIKE_LAG:
    FEATURES = [
        # Time
        "sin_hour",
        "cos_hour",
        "is_weekend",

        # Weather
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code",
        "is_day",

        # Station context
        "capacity",

        # Bike lag
        "total_bikes_lag1", 
        "total_bikes_lag2"
    ]

TARGET = "total_bikes_now"


# TRAIN MODELS PER CLUSTER

In [6]:
pred_models = {}
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n - Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    if HAS_BIKE_LAG:
        if "total_bikes_lag1" not in df_c.columns:
            df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
            df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
            df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Select train/val using the split column
    train = df_c[df_c["split"] == 0]
    val   = df_c[df_c["split"] == 1]

    X_train, y_train = train[FEATURES], train[TARGET]
    X_val, y_val     = val[FEATURES], val[TARGET]

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    pred_models[cluster_id] = {}

    # Train and evaluate on validation set
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        r2 = r2_score(y_val, preds)
        mae = mean_absolute_error(y_val, preds)

        pred_models[cluster_id][name] = model

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} -> R^2 = {r2:.3f} | MAE = {mae:.2f}")



 - Training cluster 0
  LinearRegression -> R^2 = 0.938 | MAE = 1.02
  SGDRegressor     -> R^2 = 0.938 | MAE = 1.02
  RandomForest     -> R^2 = 0.943 | MAE = 0.94

 - Training cluster 1
  LinearRegression -> R^2 = 0.968 | MAE = 1.25
  SGDRegressor     -> R^2 = 0.968 | MAE = 1.26
  RandomForest     -> R^2 = 0.970 | MAE = 1.16


# COMPARE RESULTS

In [7]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,n_samples
2,0,RandomForest,0.942562,0.935817,3433586
0,0,LinearRegression,0.937818,1.021937,3433586
1,0,SGDRegressor,0.937766,1.018985,3433586
5,1,RandomForest,0.970187,1.159925,3277252
3,1,LinearRegression,0.968102,1.253265,3277252
4,1,SGDRegressor,0.968083,1.261492,3277252


# GET BEST MODEL PER CLUSTER

In [8]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE ALL MODELS

In [9]:
for cluster_id, models_dict in pred_models.items():
    for name, model in models_dict.items():
        joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/{name}_model_cluster_{cluster_id}_{FEATURE_NAME}_{LAGGED_NAME}.pkl")

# QUICK TEST ON THE MODELS

In [11]:
# LOAD THE MODELS
pred_models = {}
for cluster_id in range(2):
    pred_models[cluster_id] = {}
    for model_name in ["LinearRegression", "SGDRegressor", "RandomForest"]:
        filename = f"{MODEL_OUTPUT_FOLDER}/{model_name}_model_cluster_{cluster_id}_{FEATURE_NAME}_{LAGGED_NAME}.pkl"
        if os.path.exists(filename):
            pred_models[cluster_id][model_name] = joblib.load(filename)
            print(f"Loaded: {filename}")
        else:
            print(f"File not found: {filename}")

Loaded: ..\data\cluster_models/LinearRegression_model_cluster_0_11_features_lagged.pkl
Loaded: ..\data\cluster_models/SGDRegressor_model_cluster_0_11_features_lagged.pkl
Loaded: ..\data\cluster_models/RandomForest_model_cluster_0_11_features_lagged.pkl
Loaded: ..\data\cluster_models/LinearRegression_model_cluster_1_11_features_lagged.pkl
Loaded: ..\data\cluster_models/SGDRegressor_model_cluster_1_11_features_lagged.pkl
Loaded: ..\data\cluster_models/RandomForest_model_cluster_1_11_features_lagged.pkl


In [12]:
df_val = df[df["split"] == 1].copy()
del df
gc.collect()

647

In [13]:
def plot_station(args):
    cluster_id, model_name, station_code, df_station = args

    df_station = df_station.sort_values("timestamp")

    X = df_station[FEATURES]
    y = df_station[TARGET]

    model = pred_models[cluster_id][model_name]
    preds = model.predict(X)

    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)

    
    """
    plt.figure(figsize=(14, 5))
    plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
    plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

    plt.title(
        f"Cluster {cluster_id} | Station {station_code}\n"
        f"Model: {model_name}"
    )
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        f"{PLOT_FOLDER}/cluster_{cluster_id}_station_{station_code}_{FEATURE_NAME}_{LAGGED_NAME}.png"
    )
    plt.close()
    """

    return {
        "cluster": cluster_id,
        "station": station_code,
        "model": model_name,
        "r2": r2,
        "mae": mae,
        "n_samples": len(df_station),
    }


In [14]:
os.cpu_count()

16

In [15]:
jobs = [
    (cluster_id, model_name, station_code, df_station)
    for (cluster_id, station_code), df_station in df_val.groupby(["cluster", "station_code"], sort=False)
    for model_name in ["LinearRegression", "SGDRegressor", "RandomForest"]
]

results = []

with ThreadPoolExecutor(max_workers=6) as ex:
    for res in tqdm(ex.map(plot_station, jobs), total=len(jobs)):
        results.append(res)

results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{PLOT_FOLDER}/all_stations_results_all_models_{FEATURE_NAME}_{LAGGED_NAME}.csv",
    index=False
)

100%|██████████| 4251/4251 [01:04<00:00, 66.16it/s]
